# Evaluate POSTER — Standalone Evaluation Notebook
Load pretrained weights từ `checkpoints/poster_best.pth`, chạy inference + đánh giá.
Không cần train lại.

In [ ]:
import os, sys, gc, random, collections, warnings
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
from torchvision import transforms
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from time import time

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
BASE_DIR = os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, 'data')
SAVE_DIR = os.path.join(BASE_DIR, 'outputs', 'evaluation')
os.makedirs(SAVE_DIR, exist_ok=True)
sys.path.insert(0, BASE_DIR)

CLASS_NAMES = ['Surprise', 'Fear', 'Disgust', 'Happiness', 'Sadness', 'Anger', 'Neutral']
NUM_CLASSES = 7
BATCH_SIZE = 16

## 1. RafDataSet — Load dữ liệu test

In [ ]:
class RafDataSet(data.Dataset):
    def __init__(self, data_path, train=True, transform=None, basic_aug=False):
        self.transform = transform
        self.train = train
        self.basic_aug = basic_aug
        csv_name = 'train_labels.csv' if train else 'test_labels.csv'
        csv_path = os.path.join(data_path, csv_name)
        df = pd.read_csv(csv_path)
        file_names = df['image'].values
        labels = df['label'].values
        self.target = np.array(labels - 1)
        split = 'train' if train else 'test'
        self.file_paths = []
        for fname, lbl in zip(file_names, labels):
            path = os.path.join(data_path, 'DATASET', split, str(lbl), fname)
            self.file_paths.append(path)
        print(f'  {"Train" if train else "Test"}: {len(self.file_paths)} samples')

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        sample = cv2.imread(self.file_paths[idx])
        if sample is None:
            sample = np.zeros((224, 224, 3), dtype=np.uint8)
        target = self.target[idx]
        if self.transform:
            sample = self.transform(sample.copy())
        return sample, target

val_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print('Loading dataset...')
test_dataset = RafDataSet(DATA_PATH, train=False, transform=val_tf)
test_loader = data.DataLoader(test_dataset, BATCH_SIZE, shuffle=False, num_workers=0)
print(f'Test batches: {len(test_loader)}')

## 2. Định nghĩa mô hình POSTER

In [ ]:
from models.ir50 import Backbone
from models.mobilefacenet import MobileFaceNet
from models.hyp_crossvit import HyVisionTransformer

def load_weights(model, ckpt):
    sd = ckpt.get('state_dict', ckpt)
    md = model.state_dict()
    new = collections.OrderedDict()
    key_map = {}
    idx = 0
    for group, count in [('body1', 3), ('body2', 4), ('body3', 14)]:
        for i in range(count):
            key_map[f'{group}.{i}.'] = f'body.{idx}.'
            idx += 1
    for k, v in sd.items():
        k_clean = k.replace('module.', '')
        mapped_k = k_clean
        for old_prefix, new_prefix in key_map.items():
            if k_clean.startswith(old_prefix):
                mapped_k = k_clean.replace(old_prefix, new_prefix)
                break
        if mapped_k in md and md[mapped_k].size() == v.size():
            new[mapped_k] = v
    md.update(new)
    model.load_state_dict(md)
    matched = len(new)
    print(f'    Loaded {matched}/{len(md)} layers')
    return model

class SE_block(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(d,d), nn.ReLU(), nn.Linear(d,d), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x)

class POSTER(nn.Module):
    def __init__(self, num_classes=7, depth=8):
        super().__init__()
        self.face_landback = MobileFaceNet([112,112], 136)
        self.ir_back = Backbone(50, 0.0, 'ir')
        self.ir_layer = nn.Linear(1024, 512)
        self.pyramid_fuse = HyVisionTransformer(
            in_chans=49, q_chanel=49, embed_dim=512,
            depth=depth, num_heads=8, mlp_ratio=2.,
            drop_rate=0., attn_drop_rate=0., drop_path_rate=0.1)
        self.se_block = SE_block(512)
        self.dropout = nn.Dropout(0.3)
        self.head = nn.Linear(512, num_classes)

    def forward(self, x):
        B = x.shape[0]
        x_face = F.interpolate(x, size=112)
        _, x_face = self.face_landback(x_face)
        x_face = x_face.view(B, -1, 49).transpose(1, 2)
        x_ir = self.ir_layer(self.ir_back(x))
        y = self.se_block(self.pyramid_fuse(x_ir, x_face))
        y = self.dropout(y)
        return self.head(y), y

## 3. Load pretrained weights

In [ ]:
CHECKPOINT_PATH = os.path.join(BASE_DIR, 'checkpoints', 'poster_best.pth')
PRETRAIN_IR50 = os.path.join(BASE_DIR, 'pretrain', 'ir50.pth')
PRETRAIN_MFN = os.path.join(BASE_DIR, 'pretrain', 'mobilefacenet_model_best.pth.tar')

print('Building POSTER model...')
model = POSTER(NUM_CLASSES, depth=8)

if os.path.exists(CHECKPOINT_PATH):
    print(f'Loading checkpoint: {CHECKPOINT_PATH}')
    ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu')
    sd = ckpt.get('state_dict', ckpt)
    # Try direct load first, fallback to load_weights
    try:
        model.load_state_dict(sd)
        print('  Direct load: OK')
    except:
        print('  Direct load failed, using load_weights mapping...')
        model = load_weights(model, sd)
else:
    print(f'Không tìm thấy {CHECKPOINT_PATH}, load pretrained backbones...')
    if os.path.exists(PRETRAIN_IR50):
        load_weights(model.ir_back, torch.load(PRETRAIN_IR50, map_location='cpu'))
    if os.path.exists(PRETRAIN_MFN):
        model.face_landback.load_state_dict(
            torch.load(PRETRAIN_MFN, map_location='cpu')['state_dict'])

model = model.to(device)
model.eval()
print('Model ready!')

## 4. Inference — Chạy dự đoán trên test set

In [ ]:
print('Running inference on test set...')
all_preds, all_labels, all_probs = [], [], []
start = time()

with torch.no_grad():
    for imgs, tgts in test_loader:
        imgs = imgs.to(device)
        logits, _ = model(imgs)
        probs = F.softmax(logits, dim=1)
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(tgts.tolist())
        all_probs.extend(probs.cpu().numpy())

P = np.array(all_preds)
T = np.array(all_labels)
probs = np.array(all_probs)
elapsed = time() - start
print(f'Done! {len(T)} samples in {elapsed:.1f}s ({len(T)/elapsed:.1f} img/s)')

## 5. Kết quả đánh giá

In [ ]:
overall_acc = (P == T).sum() / len(T) * 100
print(f'Overall Accuracy: {overall_acc:.2f}%')
print()

# Per-class accuracy
print(f'Per-Class Accuracy:')
per_class_acc = []
for i in range(NUM_CLASSES):
    mask = (T == i)
    acc = (P[mask] == i).sum() / mask.sum() * 100
    per_class_acc.append(acc)
    print(f'  {CLASS_NAMES[i]:12s}: {acc:.2f}%')
print(f'  {"Mean":12s}: {np.mean(per_class_acc):.2f}%')
print()

# Classification report
print('Classification Report:')
print(classification_report(T, P, target_names=CLASS_NAMES, digits=4))

## 6. Confusion Matrix

In [ ]:
cm = confusion_matrix(T, P)
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax[0])
ax[0].set_title('Confusion Matrix (Counts)')

cm_n = cm / cm.sum(1, keepdims=True) * 100
sns.heatmap(cm_n, annot=True, fmt='.1f', cmap='RdYlGn',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax[1])
ax[1].set_title('Confusion Matrix (%)')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix_poster.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. ROC Curves (One-vs-Rest)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve((T == i).astype(int), probs[:, i])
    auc_score = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{CLASS_NAMES[i]} (AUC={auc_score:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — POSTER')
ax.legend(loc='lower right')
plt.savefig(os.path.join(SAVE_DIR, 'roc_curves_poster.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Phân tích lỗi chi tiết

In [ ]:
errors = np.where(P != T)[0]
print(f'Total errors: {len(errors)}/{len(T)} ({len(errors)/len(T)*100:.1f}%)')
print()

# Top confused pairs
print('Top confused pairs:')
confused = []
for idx in errors:
    confused.append((CLASS_NAMES[T[idx]], CLASS_NAMES[P[idx]]))
from collections import Counter
for (true, pred), count in Counter(confused).most_common(8):
    print(f'  {true:12s} → {pred:12s}: {count} samples')

# Per-class error rate
print()
print('Per-class error rate:')
for i in range(NUM_CLASSES):
    total = (T == i).sum()
    err = ((T == i) & (P != i)).sum()
    print(f'  {CLASS_NAMES[i]:12s}: {err}/{total} ({err/total*100:.1f}%)')

## 9. Visualize mẫu bị sai

In [ ]:
# Load ảnh gốc để visualize
test_dataset_raw = RafDataSet(DATA_PATH, train=False, transform=None)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

error_indices = errors[:12]  # 12 error samples
for ax_idx, sample_idx in enumerate(error_indices):
    img, _ = test_dataset_raw[sample_idx]
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[ax_idx].imshow(img_rgb)
    axes[ax_idx].set_title(f'True: {CLASS_NAMES[T[sample_idx]]}\nPred: {CLASS_NAMES[P[sample_idx]]}')
    axes[ax_idx].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_samples_poster.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. So sánh Baseline CNN vs POSTER

In [ ]:
# Load baseline results từ notebook 3 nếu có
print('POSTER Results Summary:')
print(f'  Overall Accuracy: {overall_acc:.2f}%')
print(f'  Mean Class Acc:  {np.mean(per_class_acc):.2f}%')
print(f'  Errors:          {len(errors)}/{len(T)}')
print()
print(f'All outputs saved to: {SAVE_DIR}')

---
Hoàn tất! Tất cả biểu đồ và báo cáo đã được lưu vào `outputs/evaluation/`.